**BfPortf 02 Custom target function**

This example shows how to use [bfportf](https://github.com/answering007/bfportf) with a metric of your own:
the search optimizes a custom function, the extra metrics of the results frame stay the registry ones and
[QuantStats](https://github.com/ranaroussi/quantstats) draws the summary charts and renders the optional
HTML tearsheet.

The examples need the `examples` extra (`uv sync --extra examples`).

In [ ]:
# Import section
import urllib.request
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import quantstats as qs

import bfportf as bf

pd.set_option('display.precision', 3)

# Helper of the optional HTML tearsheet: mutes a known QuantStats matplotlib tick warning
def render_quantstats_report(portfolio, benchmark, output, title):
    """Render a QuantStats HTML tearsheet without its known matplotlib tick warning.

    QuantStats renders the yearly bar chart with ``Axes.set_xticklabels()`` but without a fixed
    locator (``quantstats/_plotting/core.py``), so matplotlib warns about possibly mislabeled
    ticks. The labels of the yearly chart are correct, so the warning is muted for this call only.
    """
    with warnings.catch_warnings():
        warnings.filterwarnings(
            'ignore',
            message=r'set_ticklabels\(\) should only be used with a fixed number of ticks',
            category=UserWarning,
        )
        qs.reports.html(portfolio, benchmark=benchmark, output=str(output), title=title)

# Helper of the charts: the portfolio and the benchmark series of a returns frame
def portfolio_and_benchmark(frame, symbols, weights, benchmark='SPY'):
    """Return the portfolio and the benchmark series of a returns frame.

    Args:
        frame (pd.DataFrame): Returns dataframe, scaled by ``weights``.
        symbols (Sequence[str]): Portfolio symbols.
        weights (pd.Series): Weights that scaled ``frame``.
        benchmark (str, optional): Benchmark symbol. Defaults to 'SPY'.

    Returns:
        tuple: Portfolio series and benchmark series, both ``float``.
    """
    portfolio = bf.get_portfolio_return(
        returns=frame, portfolio_symbols=symbols, return_weights=weights
    ).astype(float)
    benchmark_series = bf.get_portfolio_return(
        returns=frame, portfolio_symbols=[benchmark], return_weights=weights
    ).astype(float)
    return portfolio, benchmark_series


### Load data

`prices.parquet` is not tracked in git: a local copy (`assets/prices.parquet`, `prices.parquet` next to the
notebook or `../assets/prices.parquet` when the kernel runs inside `examples/`) is used when it exists,
otherwise the file is downloaded from the release asset.

In [ ]:
# Load data
# prices.parquet (4.5 MB, 500 symbols) is published as a GitHub release asset and is not tracked in git.
# A local copy is preferred: the repository layout (assets/prices.parquet), a file next to the notebook,
# or ../assets when the kernel runs inside the examples directory.
PRICES_CANDIDATES = (
    Path('assets/prices.parquet'),
    Path('prices.parquet'),
    Path('../assets/prices.parquet'),
)
PRICES_URL = 'https://github.com/answering007/bfportf/releases/latest/download/prices.parquet'

local_path = next((path for path in PRICES_CANDIDATES if path.exists()), None)

if local_path is None:
    # No local copy: take the release asset and cache it in assets/ when the directory exists
    local_path = Path('assets/prices.parquet') if Path('assets').is_dir() else Path('prices.parquet')
    try:
        with urllib.request.urlopen(PRICES_URL, timeout=120) as response:
            local_path.write_bytes(response.read())
    except Exception as error:
        raise RuntimeError(
            f'cannot download {PRICES_URL}, get the file manually: curl -L -o {local_path} {PRICES_URL}'
        ) from error
    source = f'{PRICES_URL} -> cached to {local_path}'
else:
    source = local_path

prices = pd.read_parquet(local_path)
if 'Date' in prices.columns:  # the file was written without an index
    prices = prices.set_index(pd.to_datetime(prices['Date'])).drop(columns=['Date'])

returns = bf.load_returns(prices)
bf.validate_returns(returns)
print(f'{returns.shape[0]} periods x {returns.shape[1]} symbols <- {source}')
returns.head()

### Split and scale

The split is a plain time series slice. The inverse volatility shares are estimated in-sample and the test
set is scaled with the same shares.

In [ ]:
split = int(len(returns) * 0.8)
df_train, df_test = returns.iloc[:split], returns.iloc[split:]

df_train_scaled, train_weights = bf.scale_return_data(df_train)
df_test_scaled = df_test * train_weights

print(f'Train shape: {df_train_scaled.shape}, test shape: {df_test_scaled.shape}')

### The custom metric

A metric is a callable that accepts a 1d array of returns and returns a number. The function below scores the
per period (not annualized) mean return against the per period sample standard deviation (`ddof=1`):
`((1 + mean) ** exp / (1 + std) ** (1 / exp)) ** sensitivity`. The mean is a reward, the deviation is a
divisor: a larger `exp` raises the exponent of the mean and lowers the penalty of the dispersion, so the
parameter changes which portfolio wins. The final power (`sensitivity`, 1000 by default) spreads the values
around 1, so a three decimal print shows the difference between the candidates; the power is monotone and
keeps the order of the candidates unchanged.

In [ ]:
# Custom target function: the mean is a reward, the standard deviation is a penalty,
# the final power only spreads the values that sit close to 1
def mean_std_param_target(returns: np.ndarray, exp: float = 1.0, sensitivity: int = 1000) -> float:
    """Mean and standard deviation target function.

    Scores a series by its per period (not annualized) mean return and its per period sample
    standard deviation (``ddof=1``): ``((1 + mean) ** exp / (1 + std) ** (1 / exp)) ** sensitivity``.
    The mean is a reward and the standard deviation is a divisor (a penalty), so a higher mean grows
    the value and a wider dispersion shrinks it. The values are not annualized: they are taken as is
    from the input series, so the candidates of one search stay comparable.

    Args:
        returns (np.ndarray): 1d array of returns.
        exp (float, optional): Balancing factor. Values above 1 raise the exponent of the mean and
            lower the penalty of the dispersion, so a larger ``exp`` prefers higher but more volatile
            portfolios and vice versa. Must be non zero. Defaults to 1.0.
        sensitivity (int, optional): Final power that spreads the values around 1 (1000 shows the
            difference of close candidates in the third decimal). The power is monotone, so it keeps
            the order of the candidates. Defaults to 1000.

    Returns:
        float: Target function value, larger is better (the notebook builds the ``Metric`` with
        ``maximize=True``). An all zero series gives 1.0, an empty series gives ``np.nan``.

    Raises:
        ZeroDivisionError: when ``exp`` is 0 (the ``1 / exp`` exponent).

    Notes:
        A constant series is not neutral: without a dispersion penalty the value is
        ``(1 + mean) ** sensitivity``, for example a constant 1% per period gives about 20959.

        A mean below -1 makes ``1 + mean`` negative: the score is then ``np.nan`` for a fractional
        ``exp`` and can be a misleading positive number for an even ``sensitivity``.

        The search ignores non finite values: ``np.nan`` never wins, an overflow to ``inf`` drops a
        candidate and an underflow to 0.0 makes the worst candidates tie.
    """
    value: np.float64 = (1 + returns.mean()) ** exp / (1 + returns.std()) ** (1 / exp)
    return value**sensitivity

A plain callable is accepted as well, but a `bf.Metric` also carries the name and the direction. The name
ends up in the result (`PortfolioResult.metric_name`), the direction is what the search reads instead of
guessing.

In [ ]:
# Wrap the function into a Metric: name, callable, direction
mean_std_3 = bf.Metric('mean_std_3', lambda values: mean_std_param_target(values, exp=3.0), maximize=True)

found_portfolios = bf.find_portfolios(
    returns=df_train_scaled,
    metric=mean_std_3,
    custom_weights=train_weights,
    max_number=3,
    verbose=True,
)

results = bf.results_to_frame(found_portfolios)
print(f'metric: {found_portfolios[0].metric_name} = {found_portfolios[0].value:.3f}')
results.head()

### Add the usual metrics to the results

The registry gives ready to use functions, the search direction is irrelevant here: every portfolio of the
results frame is evaluated with the same weights.

In [ ]:
for name in ('pnl', 'sharpe', 'sortino', 'max_drawdown'):
    results[name] = bf.calculate_metric(results, df_train_scaled, train_weights, bf.METRICS[name].func)

results

### The direction is a parameter

`maximize=False` flips the search: the same metric now finds the worst portfolio. `target_symbols` limits
the search to the given targets, a single symbol keeps the cell fast.

Undefined values are reported as `NaN` and candidates with a non finite value are never selected: a metric
that returns `NaN` (an empty series, for example) is skipped, the registry metrics follow the same rule (the
profit factor of a series without losing periods, for example). Only undefined values are skipped: a constant
series is a valid input for the custom metric above.

In [ ]:
worst = bf.find_portfolios(
    returns=df_train_scaled,
    metric=bf.Metric('mean_std_3', lambda values: mean_std_param_target(values, exp=3.0), maximize=False),
    custom_weights=train_weights,
    max_number=3,
    target_symbols=['SPY'],
    verbose=False,
)

worst[0].to_dict()

### QuantStats report

The cells below draw the QuantStats summary charts natively. The standalone HTML tearsheet is optional: the
commented out cells after the charts render it next to the notebook (`examples/`, also when the kernel runs
from the repository root) and print its path. The `.html` files are not tracked in git.

The charts are drawn natively instead of embedding the tearsheet, because VS Code does not render an
`<iframe>` that points to a local file
([vscode-jupyter#16398](https://github.com/microsoft/vscode-jupyter/issues/16398)). Matplotlib also warns
about `set_ticklabels()` on the yearly bar chart of the HTML report: a known QuantStats call
(`quantstats/_plotting/core.py`) without a fixed locator, the labels are correct and the
`render_quantstats_report` helper of the import cell mutes exactly this warning for the report call.

In [ ]:
# In-sample report
best = found_portfolios[0]

portfolio_series, benchmark_series = portfolio_and_benchmark(df_train_scaled, best.symbols, train_weights)

# QuantStats summary charts drawn natively: a local iframe stays blank in VS Code
qs.plots.returns(returns=portfolio_series, benchmark=benchmark_series, show=True)
qs.plots.drawdown(returns=portfolio_series, show=True)

In [ ]:
# Optional HTML tearsheet: uncomment the block to write examples/qs_in_sample.html
# report_dir = Path('examples') if Path('examples').is_dir() else Path.cwd()
# report_path = report_dir / 'qs_in_sample.html'
# render_quantstats_report(portfolio_series, benchmark_series, output=report_path, title='In-sample')
# print(f'full tearsheet: {report_path.resolve()}')

In [ ]:
# Out-of-sample report: the same symbols and the same weights on the untouched tail of the data
portfolio_series_out, benchmark_series_out = portfolio_and_benchmark(
    df_test_scaled, best.symbols, train_weights
)

# QuantStats summary charts drawn natively: a local iframe stays blank in VS Code
qs.plots.returns(portfolio_series_out, benchmark=benchmark_series_out, show=True)
qs.plots.drawdown(portfolio_series_out, show=True)

In [ ]:
# Optional HTML tearsheet: uncomment the block to write examples/qs_out_of_sample.html
# report_dir = Path('examples') if Path('examples').is_dir() else Path.cwd()
# report_path = report_dir / 'qs_out_of_sample.html'
# render_quantstats_report(portfolio_series_out, benchmark_series_out, output=report_path, title='Out-of-sample')
# print(f'full tearsheet: {report_path.resolve()}')

The custom metric is only used by the search. The extra metrics of the results frame come from the registry,
the charts come from QuantStats, so different portfolios stay comparable.

The in-sample winner is `CPRT, IEF, NVDA` (with the published `prices.parquet`): the NVDA run up of the
fitted period plus treasuries as the risk buffer, a fit artefact rather than a strategy (the out-of-sample
result is close to flat). The out-of-sample report is the honest check. Example `01` shows the single split
with a performance report, example `03` the registry, the `tolerance` and the strategies.